In [37]:
import torch
import  torch.nn as nn 
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)   # or the number of cores you have
block_size = 8
batch_size = 4 #how many blocks are running in parallel
max_iters = 500
#eval_interval = 2500
learning_rate = 3e-4
eval_iters = 50
n_embd = 384
n_head = 8
n_layer = 8
dropout = 0.2

In [38]:
chars = " "
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    chars = sorted(list(set(text)))#basically removes all repetitions and sorts them alphabetically in one array
# print(chars)
vocab_size = len(chars)

In [39]:
string_to_int = { ch:i for i, ch in enumerate(chars) }
int_to_string = { i:ch for i, ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long) #Pytorch tensor, and the datatye is torch.long (64 bit int)
print(data[:100]) #printing the first 100 integers of the tensor

tensor([92, 49, 67, 78, 70, 63, 27,  1, 33, 73, 76, 73, 78, 66, 83,  1, 59, 72,
        62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,  1, 67, 72,  1, 44, 84,
         0,  0, 30, 79, 78, 66, 73, 76, 27,  1, 41, 15,  1, 35, 76, 59, 72, 69,
         1, 31, 59, 79, 71,  0,  0, 38, 70, 70, 79, 77, 78, 76, 59, 78, 73, 76,
        27,  1, 39, 73, 66, 72,  1, 47, 15,  1, 43, 63, 67, 70, 70,  0,  0,  0,
         1,  1,  1,  1,  1,  1,  1,  1,  0, 47])


In [40]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]


#this is used to split the data into training and validation splits because we want the 
#LLM to actually learn, not just copy (which is what would happen if we gave it 100% of
#the data, and not 80-20 like we split here)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # Works if you're working on a GPU
    # x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print("inputs:")
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[ 1, 80, 63, 76, 83,  1, 74, 70],
        [59, 62, 70, 83,  0, 70, 73, 73],
        [ 1, 38,  1, 61, 59, 72, 72, 73],
        [66, 59, 78,  1, 70, 63, 78,  1]])
targets:
tensor([[80, 63, 76, 83,  1, 74, 70, 59],
        [62, 70, 83,  0, 70, 73, 73, 69],
        [38,  1, 61, 59, 72, 72, 73, 78],
        [59, 78,  1, 70, 63, 78,  1, 67]])


In [41]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [42]:
class Head(nn.Module) :

    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        
        self.dropout = nn.Dropout (dropout)
    
    def forward(self, x):
        # input ofsize (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei =q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[ :T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim =- 1) # (B, T, T)
        wei = self.dropout (wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size) :
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout (dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1) #(B,T,F) 
        out = self.dropout(self.proj(out))
        return out



class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity"""

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout (dropout),
        )
    def forward(self, x):
        return self.net(x)
        
class Block(nn.Module):

    def __init__(self, n_embd, n_head) :
    # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        y = self.sa(x)
        x = self.ln1 (x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x
        
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer) ])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, index, targets=None):
        B, T = index.shape

        # index and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(index) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)|
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
  
        return logits, loss #what are logits?

    # def generate(self, index, max_new_tokens):
    #     #index is (B, T) array of indices in the current context
    #     for _ in range(max_new_tokens):
    #         #get the predictions 
    #         logits, loss = self.forward(index)
    #         #focus only on the last time step
    #         logits = logits[:, -1, :] #becomes (B, C)
    #         #apply softmax to get probabilities
    #         probs = F.softmax(logits, dim=-1) #(B, C)
    #         #sample from the distribution
    #         index_next = torch.multinomial(probs, num_samples=1) #(B, 1)
    #         #append sampled index to the running sequence
    #         index = torch.cat((index, index_next), dim=1) #(B, T+1)
    #     return index
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
        # clamp context length to block_size
            index_cond = index[:, -block_size:]   # keep only last block_size tokens
    
            # forward pass
            logits, loss = self.forward(index_cond)
    
            # focus only on the last time step
            logits = logits[:, -1, :]  # (B, vocab_size)
    
            # softmax to get probabilities
            probs = F.softmax(logits, dim=-1)
    
            # sample from distribution
            index_next = torch.multinomial(probs, num_samples=1)
    
            # append sampled index to running sequence
            index = torch.cat((index, index_next), dim=1)

        return index


model = GPTLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


xY-OO'N(﻿C?GO:™4WF'G“—o#+Z.l8JPPV05.YOJd:Y.]V7™KCL5jU:3iGgtq(Yec?”Oor+xy"N
—y—P M4xWFTMy8P09Rx"5APp-cqqi—k?[W?13s™9lVO-aKRz9‘F‘—#)‘﻿9”Mnof—L7Ow
V™$qHa.mEOi,(i—•6/O’wg0Zx"LUG3Qd8&TG9IIL(J POAPW—*e™F1?U”XszV:$j—L2
2-L9s7﻿PUc•
’Z5ilnP(_N
)a$Lc]cOePC﻿hgKL23aGfpqhgg/&s_v9!#wu)"C•']L,t,Hs4—Lo﻿G™&—9JO4sZ,(uSDM8-iZdx]jvKf d!_Z﻿50"K0M"n6Wr(iO%--zOq3xX]fB]zE4dvIz(gz(TOOjQL™Oi—v7y”(“nIj&6OC"u4$[i396vwRxL!B_7HfHky
H™(T.O49F/_1np]!w
I—Zy•KJZ—cT%’﻿F+—u’JN*vUuuhq”sy9)piA/‘3T-bcRO—BGDwVpyCyY’n-+Cj5W wA™:™I)-—K2


In [43]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    # sample a batch of data
    xb, yb = get_batch('train')
    
    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.767, val loss: 4.758
step: 50, train loss: 2.916, val loss: 3.111
step: 100, train loss: 2.685, val loss: 2.915
step: 150, train loss: 2.651, val loss: 2.843
step: 200, train loss: 2.581, val loss: 2.772
step: 250, train loss: 2.629, val loss: 2.898
step: 300, train loss: 2.547, val loss: 2.763
step: 350, train loss: 2.618, val loss: 2.740
step: 400, train loss: 2.453, val loss: 2.718
step: 450, train loss: 2.558, val loss: 2.729
2.0283889770507812


In [44]:
#block_size = 8 this is the size of the snippet/block we're gonna be looking at. 
#it has an offset of 1, so its gonna be studying the difference between the snippets

x = train_data[:block_size]
y = train_data[1:block_size+1] #got our two snippets

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is ', context, ' target is ', target)

when input is  tensor([92])  target is  tensor(49)
when input is  tensor([92, 49])  target is  tensor(67)
when input is  tensor([92, 49, 67])  target is  tensor(78)
when input is  tensor([92, 49, 67, 78])  target is  tensor(70)
when input is  tensor([92, 49, 67, 78, 70])  target is  tensor(63)
when input is  tensor([92, 49, 67, 78, 70, 63])  target is  tensor(27)
when input is  tensor([92, 49, 67, 78, 70, 63, 27])  target is  tensor(1)
when input is  tensor([92, 49, 67, 78, 70, 63, 27,  1])  target is  tensor(33)
